In [ ]:
from IPython.display import HTML, display
display(HTML("""<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
mermaid.initialize({startOnLoad:false, theme:"neutral", securityLevel:"strict"});
await mermaid.run({nodes:document.querySelectorAll(".mermaid:not([data-processed])")});
</script>"""))


# 04b — Pydantic AI: typed compliance caseworker

## Scenario

Northstar Commerce must route payment reviews. The system may recommend **approve**, **escalate**, or **reject**, but its downstream workflow only accepts a schema-valid decision with stable evidence IDs. This makes typed dependencies and structured outputs more important than an elaborate agent loop.

<pre class="mermaid">
flowchart LR
  A["Payment case"] --> B["Screening data"]
  B --> C["Typed agent"]
  C --> D{"Schema + evidence gate"}
  D -->|"valid, low risk"| E["Queue review artifact"]
  D -->|"high risk or unsupported"| F["Escalate to analyst"]
</pre>

### Objectives

- model an agent result as a business contract;
- separate schema validation, policy validation, and evidence validation;
- identify which failures may retry and which must stop; and
- compare a deterministic fixture with optional Pydantic AI code.


## 1. Why Pydantic AI?

[Pydantic AI](https://ai.pydantic.dev/) is a Python framework built around typed agents, dependencies, tools, structured output, retries, and provider adapters. Its strongest use case is a domain where generated results become machine-consumed data: claims decisions, extracted records, triage objects, or research findings.

A valid `ComplianceDecision` means the payload is well formed. It does **not** prove the decision is factually right, that evidence belongs to the current tenant, or that payment release is authorized. Those remain deterministic application controls.


In [ ]:
from pathlib import Path
import sys
repo_root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "curriculum" / "beginner" / "04-agent-development-frameworks" / "lab.py").exists())
sys.path.insert(0, str(repo_root / "curriculum" / "beginner" / "04-agent-development-frameworks"))
from lab import *


In [ ]:
low = pydanticai_shaped_compliance_case("case-100", 450, "CA")
high = pydanticai_shaped_compliance_case("case-101", 12_500, "CA")
print(low)
print(high)
assert low.decision == "approve"
assert high.decision == "escalate" and high.requires_human_review


## 2. Validation is layered

| Layer | Example | Catches | Does not prove |
| --- | --- | --- | --- |
| Output schema | decision is an allowed literal | malformed object | factual correctness |
| Domain policy | amount >= $10k escalates | forbidden auto-approval | evidence relevance |
| Evidence gate | IDs exist for this case | invented citations | source quality |
| Authorization | reviewer can release payment | privilege misuse | model reliability |
| Evaluation | held-out cases pass | regressions | future behavior |

Use types to shrink the space of invalid states. Then put evidence checks and authorization at the action boundary.


## 3. Optional real implementation

Install `pydantic-ai`, configure a model provider, and follow the official [agents](https://ai.pydantic.dev/agents/), [output](https://ai.pydantic.dev/output/), and [testing](https://ai.pydantic.dev/testing/) guides.

```python
from dataclasses import dataclass
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext

class ReviewDecision(BaseModel):
    decision: str = Field(pattern="^(approve|escalate|reject)$")
    evidence_ids: list[str] = Field(min_length=1)
    rationale: str = Field(min_length=20)
    requires_human_review: bool

@dataclass
class CaseDeps:
    case_id: str
    amount_usd: int
    country: str

agent = Agent("openai:gpt-4.1-mini", deps_type=CaseDeps,
              output_type=ReviewDecision,
              instructions="Use screening evidence; never authorize payment release.")
```

Use an explicit provider/model that fits your deployment. Before consuming output, verify evidence IDs, tenant, freshness, and policy independently.


In [ ]:
def evidence_gate(decision, known_evidence: set[str]) -> bool:
    return bool(decision.evidence_ids) and set(decision.evidence_ids).issubset(known_evidence)

known = {"kyc-verified", "transaction-screening"}
assert evidence_gate(low, known)
assert evidence_gate(high, known)
print("Typed decision and deterministic evidence gate both passed.")


## 4. Exercises and takeaway

1. Add `tenant_id` and block cross-tenant evidence.
2. Add screening freshness and reject old screening results.
3. Test a JSON-valid decision containing an invented evidence ID.
4. Define retries: network/model failure may retry within budget; authorization or policy failure must stop.
5. Measure **supported decision rate**, not only parse success.

**Choose Pydantic AI** when typed dependencies and machine-consumed outputs are the main engineering requirement. A schema-valid answer is a necessary boundary, never sufficient authorization.
